## 环境准备：加载 Qwen 大模型

本 Notebook 使用 **ModelScope** 加载 **Qwen2.5-7B-Instruct** 模型，
替代原有的 MockLLM / 模拟 LLM，实现真实的模型推理。

> **低显存备选**：如果 GPU 显存不足，可将模型 ID 替换为 `Qwen/Qwen2.5-3B-Instruct`。

In [1]:
# ============================================================
# 安装依赖（如需要，取消注释后运行）
# ============================================================
# !pip install modelscope transformers torch -q

# ============================================================
# QwenLLM 封装类：基于 ModelScope 加载 Qwen2.5 模型
# ============================================================
from modelscope import AutoModelForCausalLM, AutoTokenizer
import torch


class QwenLLM:
    """
    基于 ModelScope 的 Qwen2.5 大模型封装类

    支持：
    - system prompt 设置
    - 多轮对话上下文维护
    - GPU / CPU 自动检测
    - 温度与生成长度控制
    """

    def __init__(self, model_name="Qwen/Qwen2.5-0.5B-Instruct", device=None):
        """
        初始化 Qwen 模型

        Args:
            model_name: 模型 ID，默认 7B；低显存可改为 "Qwen/Qwen2.5-3B-Instruct"
            device: 指定设备，None 表示自动检测
        """
        # GPU / CPU 自动检测
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = device
        print(f"[QwenLLM] 使用设备: {self.device}")
        print(f"[QwenLLM] 加载模型: {model_name}")
        print(f"[QwenLLM] 提示: 如显存不足，可替换为 Qwen/Qwen2.5-3B-Instruct")

        # 加载模型和分词器
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype="auto",
            device_map="auto"
        )
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        # 多轮对话历史
        self.messages = []

        print(f"[QwenLLM] 模型加载完成")

    def chat(self, user_message, system_prompt=None, max_new_tokens=512, temperature=0.7):
        """
        对话接口

        Args:
            user_message: 用户消息
            system_prompt: 系统提示词（可选）
            max_new_tokens: 最大生成 token 数
            temperature: 采样温度

        Returns:
            模型生成的回复文本
        """
        # 构建消息列表
        if system_prompt:
            messages = [{"role": "system", "content": system_prompt}]
        else:
            messages = []
        messages.extend(self.messages)
        messages.append({"role": "user", "content": user_message})

        # 应用聊天模板
        text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        # 生成回复
        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True
        )
        generated_ids = [
            output_ids[len(input_ids):]
            for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        response = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        print(response)
        # 更新对话历史
        self.messages.append({"role": "user", "content": user_message})
        self.messages.append({"role": "assistant", "content": response})

        return response

    def reset(self):
        """清空对话历史"""
        self.messages = []


# 初始化模型（首次运行需要下载，请耐心等待）
# llm = QwenLLM(model_name="Qwen/Qwen2.5-0.5B-Instruct")
llm = QwenLLM(model_name="Qwen/Qwen2.5-3B-Instruct")

print("\n模型就绪，可以在后续 cell 中使用 llm.chat() 进行对话")

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[QwenLLM] 使用设备: cuda
[QwenLLM] 加载模型: Qwen/Qwen2.5-3B-Instruct
[QwenLLM] 提示: 如显存不足，可替换为 Qwen/Qwen2.5-3B-Instruct


2026-06-21 14:04:02,562 - modelscope - INFO - Target directory already exists, skipping creation.
Loading weights: 100%|██████████| 434/434 [01:40<00:00,  4.32it/s]


2026-06-21 14:05:46,192 - modelscope - INFO - Target directory already exists, skipping creation.


[QwenLLM] 模型加载完成

模型就绪，可以在后续 cell 中使用 llm.chat() 进行对话


# 00 - 什么是 AI Agent？从概念到核心组件

## 学习目标

- 理解 AI Agent 的定义与核心特征
- 掌握 Agent 的四大核心组件：推理、记忆、工具、规划
- 区分 Agent 与传统 LLM 应用的本质差异
- 了解 Agent 的演进历程与发展趋势

---

## 1. AI Agent 的定义

### 1.1 什么是 Agent？

**AI Agent（人工智能代理）** 是一种能够：

1. **感知环境**（Perceive）：接收用户输入、读取文件、获取实时数据
2. **进行推理**（Reason）：分析问题、制定策略、做出决策
3. **执行行动**（Act）：调用工具、生成代码、与外部系统交互
4. **保持记忆**（Remember）：存储上下文、学习经验、积累知识

的智能系统。

> **核心定义**：Agent = LLM（大脑） + 工具（手脚） + 记忆（经验） + 规划（策略）

### 1.2 Agent vs 传统 LLM 应用

| 维度 | 传统 LLM 应用 | AI Agent |
|------|-------------|----------|
| **交互模式** | 单次问答，无状态 | 多轮对话，有状态 |
| **工具使用** | 无 | 可调用外部工具/API |
| **任务执行** | 仅生成文本 | 可执行多步骤任务 |
| **记忆能力** | 仅当前对话窗口 | 长期记忆 + 短期记忆 |
| **自主性** | 被动响应 | 主动规划与执行 |
| **错误处理** | 无 | 自我反思与修正 |

---

## 2. Agent 的核心架构

### 2.1 四大核心组件（吴恩达 Agentic Workflow 范式）

根据吴恩达在 2024 年红杉 AI 峰会的演讲，Agent 的四大核心能力为：

#### 2.1.1 反思（Reflection）

Agent 能够审视自己的输出，发现错误并进行修正。

```
用户：帮我写一段 Python 快速排序代码

Agent 初版输出：[生成代码]

Agent 自我反思：
- 这段代码是否处理了边界情况？
- 时间复杂度是否为 O(n log n)？
- 是否有更优雅的实现方式？

Agent 修正后输出：[优化后的代码]
```

#### 2.1.2 工具使用（Tool Use）

Agent 能够调用外部工具来扩展自身能力。

常见工具类型：
- **搜索工具**：Google Search、Bing Search、内部知识库检索
- **计算工具**：Python 解释器、Wolfram Alpha、计算器
- **API 工具**：天气查询、股票数据、翻译服务
- **文件工具**：读取文档、写入文件、操作数据库
- **代码工具**：执行代码、运行测试、部署服务

#### 2.1.3 规划（Planning）

Agent 能够将复杂任务分解为可执行的子任务，并制定执行计划。

```
用户：帮我策划一场 50 人的公司年会

Agent 规划：
1. 确定预算范围 → 调用财务工具查询
2. 选择场地 → 调用场地搜索工具
3. 制定活动流程 → 生成流程文档
4. 安排餐饮 → 调用餐厅预订工具
5. 发送邀请函 → 调用邮件发送工具
```

#### 2.1.4 多智能体协作（Multi-Agent Collaboration）

多个 Agent 协同工作，各自负责不同子任务。

```
项目经理 Agent：分解任务、协调进度
    ├── 研究员 Agent：收集信息、分析数据
    ├── 程序员 Agent：编写代码、调试程序
    ├── 测试员 Agent：测试功能、发现 Bug
    └── 文档员 Agent：编写文档、整理报告
```

### 2.2 Agent 架构图

```
┌─────────────────────────────────────────────────────────────┐
│                        用户输入                              │
└──────────────────────┬──────────────────────────────────────┘
                       │
                       ▼
┌─────────────────────────────────────────────────────────────┐
│                      感知层 (Perception)                     │
│         解析用户意图 · 提取关键信息 · 识别任务类型            │
└──────────────────────┬──────────────────────────────────────┘
                       │
                       ▼
┌─────────────────────────────────────────────────────────────┐
│                      规划层 (Planning)                       │
│         任务分解 · 策略制定 · 优先级排序 · 依赖分析          │
└──────────────────────┬──────────────────────────────────────┘
                       │
                       ▼
┌─────────────────────────────────────────────────────────────┐
│                      记忆层 (Memory)                         │
│    短期记忆（对话上下文）· 长期记忆（知识库）· 工作记忆        │
└──────────────────────┬──────────────────────────────────────┘
                       │
                       ▼
┌─────────────────────────────────────────────────────────────┐
│                      推理层 (Reasoning)                      │
│         LLM 核心 · Chain-of-Thought · ReAct · Reflection     │
└──────────────────────┬──────────────────────────────────────┘
                       │
                       ▼
┌─────────────────────────────────────────────────────────────┐
│                      行动层 (Action)                         │
│         工具调用 · API 请求 · 代码执行 · 文件操作            │
└──────────────────────┬──────────────────────────────────────┘
                       │
                       ▼
┌─────────────────────────────────────────────────────────────┐
│                      输出层 (Output)                         │
│         生成回复 · 返回结果 · 请求澄清 · 报告进度            │
└─────────────────────────────────────────────────────────────┘
```

---

## 3. Agent 的演进历程

### 3.1 发展阶段

| 阶段 | 时间 | 特征 | 代表技术 |
|------|------|------|----------|
| **阶段 1：Prompt Engineering** | 2022-2023 | 精心设计的提示词，单次调用 | Few-shot、CoT prompting |
| **阶段 2：Chain/Flow** | 2023 | 固定流程链，预定义步骤 | LangChain Chains、LLMChain |
| **阶段 3：Tool-Augmented** | 2023-2024 | LLM + 工具调用能力 | ReAct、Toolformer、GPT-4 Plugins |
| **阶段 4：Autonomous Agent** | 2024 | 自主规划、执行、反思 | AutoGPT、BabyAGI、AgentGPT |
| **阶段 5：Multi-Agent System** | 2024-2025 | 多 Agent 协作、角色分工 | AutoGen、CrewAI、MetaGPT |
| **阶段 6：Agentic Workflow** | 2025+ | 企业级、生产级 Agent 系统 | LangGraph、Google ADK、企业 Agent 平台 |

### 3.2 关键里程碑

- **2022.11**：ChatGPT 发布，展示对话能力
- **2023.03**：ReAct 论文（Reasoning + Acting）被广泛采用
- **2023.04**：AutoGPT 发布，引发自主 Agent 热潮
- **2023.10**：LangChain 成为主流开发框架
- **2024.03**：Claude 3 发布，工具调用能力大幅提升
- **2024.06**：AutoGen v0.4 发布，多 Agent 协作成熟
- **2024.09**：LangGraph 稳定版发布，状态图工作流
- **2025.01**：Google ADK 发布，生产级 Agent 框架
- **2025.04**：OpenAI Agents SDK 发布

---

## 4. Agent 的分类体系

### 4.1 按能力层级分类

```
Level 0: 简单 Agent（Simple Agent）
    └── 单次调用，无状态，如基础问答机器人

Level 1: 工具增强 Agent（Tool-Augmented Agent）
    └── 可调用工具，有短期记忆，如 ReAct Agent

Level 2: 自主 Agent（Autonomous Agent）
    └── 自主规划、执行、反思，如 AutoGPT

Level 3: 多智能体系统（Multi-Agent System）
    └── 多 Agent 协作，角色分工，如 CrewAI

Level 4: 组织级 Agent（Organizational Agent）
    └── 企业级部署，与人类协作，持续学习
```

### 4.2 按应用场景分类

| 类型 | 描述 | 示例 |
|------|------|------|
| **对话型 Agent** | 专注于多轮对话交互 | 客服机器人、个人助手 |
| **任务型 Agent** | 完成特定任务 | 代码生成、数据分析 |
| **研究型 Agent** | 信息收集与分析 | 市场调研、文献综述 |
| **创作型 Agent** | 内容生成与编辑 | 写作助手、设计助手 |
| **协作型 Agent** | 团队协作支持 | 项目管理、会议助理 |

---

## 5. 动手实践：构建最简 Agent

让我们用纯 Python 实现一个最简化的 Agent，理解其核心机制。

In [2]:
# ============================================================
# 最简化 Agent 实现（已集成 QwenLLM）
# ============================================================

# 最简化 Agent 实现
# 这个 Agent 具备：感知 → 推理 → 行动 → 记忆 的完整循环

class SimpleAgent:
    """最简 Agent 实现"""
    
    def __init__(self, name="SimpleAgent"):
        self.name = name
        self.memory = []  # 短期记忆：对话历史
        self.tools = {}   # 可用工具注册表
    
    def register_tool(self, name, func, description):
        """注册工具"""
        self.tools[name] = {
            "func": func,
            "description": description
        }
        print(f"[系统] 工具 '{name}' 已注册: {description}")
    
    def perceive(self, user_input):
        """感知层：接收并理解用户输入"""
        print(f"\n[感知] 用户输入: {user_input}")
        
        # 简单意图识别
        if "天气" in user_input:
            return {"intent": "query_weather", "input": user_input}
        elif "计算" in user_input or "等于" in user_input:
            return {"intent": "calculate", "input": user_input}
        else:
            return {"intent": "chat", "input": user_input}
    
    def reason(self, perception):
        """推理层：使用 QwenLLM 根据感知结果进行推理"""
        intent = perception["intent"]
        print(f"[推理] 识别意图: {intent}")

        # 使用真实 LLM 进行推理决策
        prompt = f"用户意图是 {intent}，输入内容：{perception['input']}。"
        if intent == "query_weather":
            prompt += '请从消息中提取城市名，输出 JSON：{"action": "get_weather", "params": {"city": "城市名"}}'
        elif intent == "calculate":
            prompt += '请提取数学表达式，输出 JSON：{"action": "calculate", "params": {"expression": "表达式"}}'
        else:
            prompt += '请输出 JSON：{"action": "respond", "params": {"message": "你的回复"}}'

        response = llm.chat(
            prompt,
            system_prompt="你是一个 Agent 推理模块，只输出 JSON，不要其他文字。",
            max_new_tokens=128,
            temperature=0.3
        )

        # 解析 LLM 输出
        try:
            import json as _json
            import re
            json_match = re.search(r'\{.*\}', response, re.DOTALL)
            if json_match:
                return _json.loads(json_match.group())
        except:
            pass

        # 解析失败时使用默认策略（无模型时的备选方案）
        # if intent == "query_weather":
        #     return {"action": "get_weather", "params": {"city": "北京"}}
        # elif intent == "calculate":
        #     return {"action": "calculate", "params": {"expression": "2+2"}}
        # else:
        #     return {"action": "respond", "params": {"message": response}}
    
    def act(self, decision):
        """行动层：执行决策"""
        action = decision["action"]
        params = decision["params"]
        
        print(f"[行动] 执行: {action}")
        
        if action in self.tools:
            result = self.tools[action]["func"](**params)
            return result
        elif action == "respond":
            return params["message"]
        else:
            return f"未知动作: {action}"
    
    def remember(self, user_input, result):
        """记忆层：存储交互历史"""
        self.memory.append({"input": user_input, "output": result})
        # 只保留最近 5 轮对话
        if len(self.memory) > 5:
            self.memory.pop(0)
    
    def run(self, user_input):
        """运行 Agent 完整循环"""
        # 1. 感知
        perception = self.perceive(user_input)
        
        # 2. 推理
        decision = self.reason(perception)
        
        # 3. 行动
        result = self.act(decision)
        
        # 4. 记忆
        self.remember(user_input, result)
        
        print(f"[输出] {result}")
        return result

# 定义工具函数
def get_weather(city):
    """模拟天气查询工具"""
    weather_data = {
        "北京": "晴天，25°C",
        "上海": "多云，28°C",
        "深圳": "小雨，30°C"
    }
    return weather_data.get(city, f"未找到 {city} 的天气信息")

def calculate(expression):
    """计算工具"""
    try:
        # 注意：实际生产环境需要更安全的计算方式
        result = eval(expression)
        return f"计算结果: {result}"
    except:
        return "计算失败，请检查表达式"

# 创建 Agent 实例
agent = SimpleAgent(name="小助手")

# 注册工具
agent.register_tool("get_weather", get_weather, "查询指定城市的天气")
agent.register_tool("calculate", calculate, "执行数学计算")

[系统] 工具 'get_weather' 已注册: 查询指定城市的天气
[系统] 工具 'calculate' 已注册: 执行数学计算


In [3]:
# 测试 Agent
agent.run("今天北京的天气怎么样？")


[感知] 用户输入: 今天北京的天气怎么样？
[推理] 识别意图: query_weather
{"action": "get_weather", "params": {"city": "北京"}}
[行动] 执行: get_weather
[输出] 晴天，25°C


'晴天，25°C'

In [4]:
agent.run("帮我计算 15 * 23 等于多少？")


[感知] 用户输入: 帮我计算 15 * 23 等于多少？
[推理] 识别意图: calculate
{"action": "calculate", "params": {"expression": "15 * 23"}}
[行动] 执行: calculate
[输出] 计算结果: 345


'计算结果: 345'

In [5]:
agent.run("你好，很高兴认识你！")


[感知] 用户输入: 你好，很高兴认识你！
[推理] 识别意图: chat
{"action": "respond", "params": {"message": "你好，很高兴认识你！"}}
[行动] 执行: respond
[输出] 你好，很高兴认识你！


'你好，很高兴认识你！'

---

## 6. Agent 设计模式概览

本教程后续将深入讲解以下核心设计模式：

| 模式 | 核心思想 | 适用场景 |
|------|----------|----------|
| **ReAct** | 推理与行动交替进行 | 需要多步推理的复杂任务 |
| **CoT (Chain-of-Thought)** | 显式展示思考过程 | 数学推理、逻辑分析 |
| **Reflection** | 自我审视与修正 | 代码生成、内容创作 |
| **Plan-and-Execute** | 先规划后执行 | 多步骤任务、项目管理 |
| **Tool Use** | 调用外部工具扩展能力 | 实时数据查询、计算 |
| **Multi-Agent** | 多 Agent 协作分工 | 复杂项目、团队协作 |

---

## 7. 小结

### 核心要点

1. **Agent 的本质**：LLM + 工具 + 记忆 + 规划
2. **四大核心能力**：反思、工具使用、规划、多智能体协作
3. **关键差异**：Agent 具有自主性、状态保持和工具调用能力
4. **演进趋势**：从简单问答 → 工具增强 → 自主执行 → 多 Agent 协作

### 下一步

- [01_react_and_cot.ipynb](01_react_and_cot.ipynb) - 深入学习 ReAct 和 CoT 推理模式
- [02_tool_use_and_function_calling.ipynb](02_tool_use_and_function_calling.ipynb) - 掌握工具调用与 Function Calling
- [03_memory_systems.ipynb](03_memory_systems.ipynb) - 理解 Agent 记忆系统设计

---

## 参考资源

- [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629)
- [Chain-of-Thought Prompting Elicits Reasoning in LLMs](https://arxiv.org/abs/2201.11903)
- [吴恩达 Agentic Workflow 演讲](https://www.deeplearning.ai/)
- [LangChain Agent 文档](https://python.langchain.com/docs/modules/agents/)